In [117]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from typing import Counter

In [147]:
import seaborn as sns

df = sns.load_dataset("penguins")

print(df.head())

  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
3  Adelie  Torgersen             NaN            NaN                NaN   
4  Adelie  Torgersen            36.7           19.3              193.0   

   body_mass_g     sex  
0       3750.0    Male  
1       3800.0  Female  
2       3250.0  Female  
3          NaN     NaN  
4       3450.0  Female  


In [148]:
# Example

#dataset = np.array([
#    [10, 0],
#    [12, 0],
#    [14, 0],
#    [16, 1],
#    [18, 1],
#    [20, 1],
#    [22, 1],
#    [24, 1]
#])


#left, right = splits(dataset, feature=0, threshold=14)
#gain = information_gain(dataset,left, right)

#print(left)
#print(right)
#print(gain) # 0.95 sin the father's entropy is 95 

In [149]:
class Node():

    def __init__(self,feature=None, threshold=None, left=None, right=None, gain=None, value=None):

        self.feature = feature
        self.threshold = threshold
        self.left= left
        self.right = right
        self.gain = gain
        self.value = value

class Decisiontree():

    def __init__(self, min_samples=2, max_depth=2):
        self.min_samples = min_samples
        self.max_depth = max_depth


    def entropy(self, class_p):
        return sum(-p* math.log(p,2) for p in class_p if p > 0 )

    def class_prob(self, lables):
        '''
        X / total(variables)
        e.g ['a','a','a','a','b','b','c']
        'a' = 4
        4 / 7
        '''
        total = len(lables)
        return [count / total for count in Counter(lables).values()]

    def data_entropy(self, labels:list):
        '''
        -SUM(X * LOG2(X))
        '''
        return self.entropy(self.class_prob(labels))

    def splits(self, df, feature, threshold):
        right = []
        left = []

        for row in df:
            if row[feature] <= threshold:
                left.append(row)
            else:
                right.append(row)

        right = np.array(right)
        left = np.array(left)

        return left, right

    def information_gain(self, main, left, right):

        main = main[:,-1] 
        left = left[:,-1]
        right = right[:,-1]

        main_entropy = self.data_entropy(main)
        left_weight = len(left) / len(main)
        right_weight = len(right) / len(main)

        l_entropy, r_entropy = self.data_entropy(left), self.data_entropy(right)

        weighted_entropy = left_weight * l_entropy + right_weight * r_entropy
        i_gain = main_entropy - weighted_entropy

        return i_gain

    def best_split(self, df, n_samples, n_features):
        df = df.copy()
        info_splits = {'gain':0,
                    'feature':None,
                    'threshold':None,
                    'left_df':None,
                    'right_df':None}

        for indx_feature in range(n_features):
            features = df[:,indx_feature]
            thresholds = np.unique(features)

            for threshold in thresholds:
                lf_df , rg_df = self.splits(df, indx_feature, threshold)

                if len(lf_df) and len(rg_df):

                    info_gain = self.information_gain(df, lf_df, rg_df)

                    if info_gain >= info_splits['gain']:
                        info_splits['feature'] = indx_feature
                        info_splits['threshold'] = threshold
                        info_splits['left_df'] = lf_df
                        info_splits['right_df'] = rg_df
                        info_splits['gain'] = info_gain
        return info_splits

    def leaf_value(self, y):
        y = list(y)
        most_seen_value = max(y, key=y.count)

        return most_seen_value

    def tree(self, df, current_depth=0):

        X, y = df[:,:-1], df[:,-1]
        n_samples, n_features = X.shape
        if n_samples >= self.min_samples and current_depth < self.max_depth:
            best_split = self.best_split(df, n_samples, n_features)

            if best_split['gain']:
                left_n = self.tree(best_split['left_df'], current_depth+1)
                right_n = self.tree(best_split['right_df'], current_depth+1)

                return Node(best_split['feature'],best_split['threshold'] ,left_n, right_n, best_split['gain'])

        return Node(value=self.leaf_value(y))

    def fit(self,X, y):
        df = np.concatenate((X,y.values.reshape(-1,1)),axis=1)
        self.root = self.tree(df)

    def prediction(self, X, node):

        if node.value is not None:
            return node.value

        if X[node.feature] <= node.threshold:
            return self.prediction(X, node.left)
        return self.prediction(X, node.right)

    def predict(self, X):
        return np.array([self.prediction(x, self.root) for x in np.asarray(X)])


In [150]:
df.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female


In [151]:
modelo_tree = Decisiontree(2,2)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,classification_report)
from sklearn.tree import DecisionTreeClassifier

df['species'] = df['species'].map({'Adelie':1,'Gentoo':0,'Chinstrap':2})
df = pd.concat([df.drop(columns=['island']),pd.get_dummies(df['island'],dtype=int)],axis=1)
df['sex'] = df['sex'].map({'Male':1,'Female':0})



X, y = df.drop(columns=['species']), df['species']
X.columns = [str(c) for c in X.columns] 

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)

modelo_tree.fit(X_train,y_train)
predictions = modelo_tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions)) 

Accuracy: 0.927536231884058
[[20  0  1]
 [ 0 29  3]
 [ 0  1 15]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.98        21
           1       0.97      0.91      0.94        32
           2       0.79      0.94      0.86        16

    accuracy                           0.93        69
   macro avg       0.92      0.93      0.92        69
weighted avg       0.94      0.93      0.93        69



In [152]:
modelo_sk = DecisionTreeClassifier()

modelo_sk.fit(X_train,y_train)

predictions_sk = modelo_sk.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions_sk))
print(confusion_matrix(y_test, predictions_sk))
print(classification_report(y_test, predictions_sk)) 

Accuracy: 0.9565217391304348
[[20  1  0]
 [ 0 31  1]
 [ 0  1 15]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.98        21
           1       0.94      0.97      0.95        32
           2       0.94      0.94      0.94        16

    accuracy                           0.96        69
   macro avg       0.96      0.95      0.96        69
weighted avg       0.96      0.96      0.96        69

